Code that takes the ansatz state circuits from running the algorithms in the code Schwinger_CD_algorithms.ipynb and transpiles that through the fake IBM backend FakeAlgiers (https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider-fake-algiers) and counts the number of specific operations, the total number of operations, and the depth of the circuit and saves the data into a text file.

The possible operations for FakeAlgiers are CX, Id, RZ, SX, and X gates, and delay, reset and measure operations.

This code is currently set to do this resource estimate for the states from the best runs with the fixed A_DC terms from running Schwinger_CD_algorithms.ipynb with 4, 6, 8 and 10 qubits and for both X and XY mixers. The model parameter values, variational parameter values, and fixed A_DC terms are needed to recreate these states. This data is taken from the text files outputs of Schwinger_CD_algorithms.ipynb.

Saves the data in a text file. If you do not want the text files saved comment out those sections

With the current variables set as they are, the code will take about a minute to run.

In [ ]:
# Imports the modules

import numpy as np
from scipy.optimize import minimize
from datetime import datetime
 
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.parametervector import ParameterVector
from qiskit.circuit.quantumcircuit import QuantumCircuit

In [ ]:

# Function that creates the QAOA ansatz state circuit, transpiles through FakeAlgiers, and then counts the number of specific operations,
# the total number of operators, and the depth of the circuit.
# Inputs problem Hamiltonian (H_prob), mixer Hamiltonian (H_mix), mixer ground state (initial_state), number of layers in circuit (layer), 
# number of qubits (num_qubits), and variational parameter values (parameters).
# Returns the counts of the different operations (count_ops), total number of operations (size), and the circuit depth (depth).

def QAOA(H_prob, H_mix, initial_state, layers, num_qubits, parameters):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layers)
    betas  = ParameterVector('β', layers)
    
    # Builds the ansatz state circuit
    QAOA_Ansatz = initial_state.copy()
    for i in range(layers):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        QAOA_Ansatz.append(gate_prob, range(num_qubits))
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        QAOA_Ansatz.append(gate_mix, range(num_qubits))

    # Seeding the algorithm for better testing.
    seed = 100 
    # Create pass manager for transpilation
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed)

    # Assigns the variational parameter values.
    QAOA_Ansatz = QAOA_Ansatz.assign_parameters(parameters)
    candidate_circuit = pm.run(QAOA_Ansatz)
    # Counts the number of specific operations.
    count_ops = candidate_circuit.count_ops()
    # Counts the total number of operations.
    size = candidate_circuit.size()
    # Counts the circuit depth.
    depth = candidate_circuit.depth()
    
    return count_ops, size, depth

In [ ]:

# Function that creates the DC-QAOA ansatz state circuit, transpiles through fakeAlgiers, and then counts the number of specific operations,
# the total number of operators, and the depth of the circuit.
# Inputs problem Hamiltonian (H_prob), mixer Hamiltonian (H_mix), A_DC operators (A_DC), mixer ground state (initial_state), 
# number of layers in circuit (layer), number of qubits (num_qubits), and variational parameter values (parameters).
# Returns the counts of the different operations (count_ops), total number of operations (size), and the circuit depth (depth).

def DC_QAOA(H_prob, H_mix, A_DC, initial_state, layers, num_qubits, parameters):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layers)
    betas  = ParameterVector('β', layers)
    alphas = ParameterVector('α', layers)
    
    # Builds the ansatz state circuit.
    DC_QAOA_Ansatz = initial_state.copy()
    for i in range(layers):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        DC_QAOA_Ansatz.append(gate_prob, range(num_qubits))
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        DC_QAOA_Ansatz.append(gate_mix, range(num_qubits))
        # Applys A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        DC_QAOA_Ansatz.append(gate_CD, range(num_qubits))

    # Seeding the algorithm for better testing
    seed = 100 
    # Create pass manager for transpilation.
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed)

    # Assigns the variational parameter values.
    DC_QAOA_Ansatz = DC_QAOA_Ansatz.assign_parameters(parameters)
    candidate_circuit = pm.run(DC_QAOA_Ansatz)
    # Counts the number of specific operations.
    count_ops = candidate_circuit.count_ops()
    # Counts the total number of operations.
    size = candidate_circuit.size()
    # Counts the circuit depth.
    depth = candidate_circuit.depth()
    
    return count_ops, size, depth

In [ ]:

# Function that creates the CD-inspired ansatz state circuit, transpiles through fakeAlgiers, and then counts the number of specific operations,
# the total number of operators, and the depth of the circuit.
# Inputs problem Hamiltonian (H_prob), A_DC operators (A_DC), mixer ground state (initial_state), number of layers in circuit (layer), 
# number of qubits (num_qubits), and variational parameter values (parameters).
# Returns the counts of the different operations (count_ops), total number of operations (size), and the circuit depth (depth).

def CD_inspired(H_prob, A_DC, initial_state, layers, num_qubits, parameters):
    
    # Defines the sets of variational parameters.
    alphas = ParameterVector('α', layers)
    
    # Builds the ansatz state circuit.
    CD_inspired_Ansatz = initial_state.copy()
    for i in range(layers):
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_inspired_Ansatz.append(gate_CD, range(num_qubits))

    # Seeding the algorithm for better testing
    seed = 100 
    # Create pass manager for transpilation
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed)

    # Assigns the variational parameter values.
    CD_inspired_Ansatz = CD_inspired_Ansatz.assign_parameters(parameters)
    candidate_circuit = pm.run(CD_inspired_Ansatz)
    # Counts the number of specific operations.
    count_ops = candidate_circuit.count_ops()
    # Counts the total number of operations.
    size = candidate_circuit.size()
    # Counts the circuit depth.
    depth = candidate_circuit.depth()
    
    return count_ops, size, depth

In [ ]:

# Function that creates the CD-mixer ansatz state circuit, transpiles through fakeAlgiers, and then counts the number of specific operations,
# the total number of operators, and the depth of the circuit.
# Inputs mixer Hamiltonian (H_mix), A_DC operators (A_DC), mixer ground state (initial_state), number of layers in circuit (layer), 
# number of qubits (num_qubits), and variational parameter values (parameters).
# Returns the counts of the different operations (count_ops), total number of operations (size), and the circuit depth (depth).

def CD_mixer(H_prob, H_mix, A_DC, initial_state, layers, num_qubits, parameters):
    
    # Defines the sets of variational parameters.
    betas  = ParameterVector('β', layers)
    alphas = ParameterVector('α', layers)
    
    # Building the circuit.
    CD_mixer_Ansatz = initial_state.copy()
    for i in range(layers):
        # Applys mixer Hamiltonian unitary.
        gate_mix = PauliEvolutionGate(H_mix, time=betas[i])
        CD_mixer_Ansatz.append(gate_mix, range(num_qubits))
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_mixer_Ansatz.append(gate_CD, range(num_qubits))

    # Seeding the algorithm for better testing
    seed = 100 
    # Create pass manager for transpilation
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed)

    # Assigns the variational parameter values.
    CD_mixer_Ansatz = CD_mixer_Ansatz.assign_parameters(parameters)
    candidate_circuit = pm.run(CD_mixer_Ansatz)
    # Counts the number of specific operations.
    count_ops = candidate_circuit.count_ops()
    # Counts the total number of operations.
    size = candidate_circuit.size()
    # Counts the circuit depth.
    depth = candidate_circuit.depth()
    
    return count_ops, size, depth

In [ ]:

# Function that creates the CD-prob ansatz state circuit, transpiles through fakeAlgiers, and then counts the number of specific operations,
# the total number of operators, and the depth of the circuit.
# Inputs problem Hamiltonian (H_prob), A_DC operators (A_DC), mixer ground state (initial_state), number of layers in circuit (layer), 
# number of qubits (num_qubits), and variational parameter values (parameters).
# Returns the counts of the different operations (count_ops), total number of operations (size), and the circuit depth (depth).

def CD_prob(H_prob, A_DC, initial_state, layers, num_qubits, parameters):
    
    # Defines the sets of variational parameters.
    gammas = ParameterVector('γ', layers)
    betas  = ParameterVector('β', layers)
    alphas = ParameterVector('α', layers)
    
    # Building the circuit.
    CD_prob_Ansatz = initial_state.copy()
    for i in range(layers):
        # Applys problem Hamiltonian unitary.
        gate_prob = PauliEvolutionGate(H_prob, time=gammas[i])
        CD_prob_Ansatz.append(gate_prob, range(num_qubits))
        # Applys the A_DC unitary.
        gate_CD = PauliEvolutionGate(A_DC[i], time=alphas[i])
        CD_prob_Ansatz.append(gate_CD, range(num_qubits))

    # Seeding the algorithm for better testing
    seed = 100 
    # Create pass manager for transpilation
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=seed)

    # Assigns the variational parameter values.
    CD_prob_Ansatz = CD_prob_Ansatz.assign_parameters(parameters)
    candidate_circuit = pm.run(CD_prob_Ansatz)
    # Counts the number of specific operations.
    count_ops = candidate_circuit.count_ops()
    # Counts the total number of operations.
    size = candidate_circuit.size()
    # Counts the circuit depth.
    depth = candidate_circuit.depth()
    
    return count_ops, size, depth

In [ ]:

# Defines the Schwinger model Hamiltonian for a given J, w, m, theta.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# Outputs the Hamiltonian.

def Schwinger_Hamiltonian(num_qubits, J, w, m, theta):

    # Constructs the H_ZZ term.
    Paulis_ZZ = []
    for n in range(1, num_qubits):
        for k in range(1,n):
            Paulis_ZZ.append(("ZZ", [k-1, n-1], J*(num_qubits-n)/2))
    
    H_ZZ = SparsePauliOp.from_sparse_list(Paulis_ZZ, num_qubits)

    # Constructs the H_+- term.
    Paulis_pm = []
    for n in range(1,num_qubits):
        W_n = w-(-1)**n*m*np.sin(theta)/2
        Paulis_pm.append(("XX", [n-1, n], W_n/2))
        Paulis_pm.append(("YY", [n-1, n], W_n/2))
    
    H_pm = SparsePauliOp.from_sparse_list(Paulis_pm, num_qubits)

    # Constructs the H_Z term.
    Paulis_Z = []
    for n in range(1, num_qubits):
        C_n = np.floor(num_qubits/2) - np.floor(n/2)
        Paulis_Z.append(("Z", [n-1], (-1)**n*m*np.cos(theta)/2-J*C_n/2))
    Paulis_Z.append(("Z", [num_qubits-1], (-1)**num_qubits*m*np.cos(theta)/2))
    
    H_Z = SparsePauliOp.from_sparse_list(Paulis_Z, num_qubits)

    H_prob_Schwinger = H_ZZ + H_pm + H_Z
    # Removes terms from the Hamiltonian with very small coefficients which are meant to be zero as this can lead to errors in the optimizer.
    H_prob_Schwinger = H_prob_Schwinger.simplify(atol=1e-10)
    
    return H_prob_Schwinger

In [ ]:

# Constructs and returns an XY mixer Hamiltonian and its ground state to be used as the initial state in the ansatz state of the algorithms.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def XY_mixer(num_qubits):

    # Constructing the XY mixer Hamiltonian.
    Paulis_mix = []
    for n in range(num_qubits-1):
        Paulis_mix.append(("XX", [n,n+1], 1/2))
        Paulis_mix.append(("YY", [n,n+1], 1/2))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)

    # Constructing the ground state of the XY mixer Hamiltonian by numerically finding the ground state and then using .initialize to
    # initialize a quantum circuit to that state.
    initial_state = QuantumCircuit(num_qubits)
    H_mix_matrix = np.array(H_mix)
    eigenvalues, eigenvectors = np.linalg.eigh(H_mix_matrix)
    initial_state.initialize(eigenvectors[:,0], initial_state.qubits)

    return H_mix, initial_state

In [ ]:

# Constructs and returns an X mixer Hamiltonian and its ground state to be used as the initial state in the ansatz state of the algorithms.
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.

def X_mixer(num_qubits):

    # Constructing the X mixer Hamiltonian
    Paulis_mix = []
    for i in range(num_qubits):
        Paulis_mix.append(("X", [i], 1))
    
    H_mix = SparsePauliOp.from_sparse_list(Paulis_mix, num_qubits)
    
    # Creates the ground state of the X mixer Hamiltonian
    initial_state = QuantumCircuit(num_qubits)
    initial_state.h(range(num_qubits))
    initial_state.z(range(num_qubits))
    
    return H_mix, initial_state

In [ ]:

# Constructs and returns an A_DC operator from the operator pool A_XY generated by calculating A_λ for the Schwinger model Hamiltonian with an XY mixer 
# with ℓ=2 and keeping up to three-body terms.
# The operator pool is A_XY = ['XY', 'YX', 'XYZ', 'XZY', 'YXZ', 'YZX', 'ZXY', 'ZYX'].
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# The input 'term' determine which operator from the pool is used. E.g. to use the term XY from A_XY, input 'XY' into the function.
# The chosen operator term is summed over all the qubits while preserving the ordering of the operator term.

def A_DC_XY(num_qubits, term):

    # Gives an error message if a operator term entered is not in A_XY.
    if term!='XY' and term!='YX' and term!='ZXY' and term!='ZYX' and term!='XZY' and term!='XYZ' and term!='YZX' and term!='YXZ':
        print(term, 'is not a valid CD term for this Hamiltonian.')
        return

    Paulis_CD = []
    # Constructs A_DC for a two-bodied operator.
    if term=='XY' or term=='YX':
        for n in range(1, num_qubits+1):
            for k in range(1,n):
                Paulis_CD.append((term, [k-1, n-1], 1))

    # Constructs A_DC for a three-bodied operator.
    else:
        for a in range(1, num_qubits+1):
            for b in range(1,a):
                for c in range(1,b):
                    Paulis_CD.append((term, [c-1, b-1, a-1], 1))
       
    A_DC = SparsePauliOp.from_sparse_list(Paulis_CD, num_qubits)
    
    return A_DC

In [ ]:

# Constructs and returns an A_DC operator from the operator pool A_X generated by calculating A_λ for the Schwinger model Hamiltonian with an X mixer 
# with ℓ=2 and keeping up to two-body terms.
# The operator pool is A_X = ['Y', 'XY', 'YX', 'YZ', 'ZY'].
# Uses SparsePauliOp from qiskit. First generates a list of Pauli operators and then uses .from_sparse_list to generate the sparse Pauli operator.
# The input 'term' determine which operator from the pool is used. E.g. to use the term XY from A_X, input 'XY' into the function.
# The chosen operator term is summed over all the qubits while preserving the ordering of the operator term.

def A_DC_X(num_qubits, term):

    # Gives an error message if a operator term entered is not in A_XY.
    if term!='Y' and term!='YZ' and term!='ZY' and term!='XY' and term!='YX':
        print(term, 'is not a valid CD term for this Hamiltonian.')
        return

    Paulis_CD = []
    # Constructs A_DC for a one-bodied operator.
    if term=='Y':
        for i in range(num_qubits):
            Paulis_CD.append(("Y", [i], 1))

    # Constructs A_DC for a two-bodied operator.
    else:
        for n in range(1, num_qubits+1):
            for k in range(1,n):
                Paulis_CD.append((term, [k-1, n-1], 1))

    A_DC = SparsePauliOp.from_sparse_list(Paulis_CD, num_qubits)
    
    return A_DC

In [ ]:

# Sets the variables the circuits.

# Sets the number of layers to create the circuits with.
layers = np.arange(1,6)
# Sets the different numbers of qubits to create the circuits with.
num_qubits_list = [4, 6, 8, 10]

# Model parameters for Schwinger model.
a = 5/6
a_string = '5/6' # Saved as a string so that the exact fraction appears in plot title and text file.
g = 1
m = 0.35
theta = 0
J = g**2*a/2
w = 1/(2*a)

# The A_DC data taken from text files produced from running Schwinger_CD_algorithms.ipynb.
# For the circuits with an XY mixer.
A_DC_term_DC_QAOA_XY = [['XY', 'YX', 'YX', 'XZY', 'YX'],['YX', 'XY', 'YX', 'YX', 'XY'],['YX', 'XY', 'XY', 'XY', 'YX'],['YX', 'XY', 'YX', 'YX', 'XY']]
A_DC_term_CD_inspired_XY = [['XY', 'XYZ', 'XY', 'XY', 'XY'],['XY', 'YX', 'XY', 'XY', 'YX'],['XY', 'XY', 'YX', 'XY', 'YX'],['XY', 'YX', 'XY', 'XY', 'XY']]
A_DC_term_CD_mixer_XY = [['YX', 'XY', 'YX', 'XY', 'YZX'],['XY', 'XY', 'YX', 'YX', 'YX'],['XY', 'YX', 'XY', 'YX', 'YX'],['YX', 'XY', 'YX', 'YX', 'YX']]
A_DC_term_CD_prob_XY = [['XY', 'ZXY', 'XY', 'ZXY', 'YX'], ['XY', 'XY', 'YX', 'YX', 'XY'],['XY', 'YX', 'XY', 'YX', 'YX'],['YX', 'XY', 'YX', 'YX', 'XY']]

# For the circuits with an X mixer.
A_DC_term_DC_QAOA_X = [['YX', 'ZY', 'XY', 'Y', 'YX'],['XY', 'Y', 'YX', 'YX', 'XY'],['YX', 'YX', 'Y', 'YX', 'Y'],['XY', 'Y', 'Y', 'Y', 'Y']]
A_DC_term_CD_inspired_X = [['YX', 'ZY', 'YX', 'YX', 'ZY'],['YX', 'ZY', 'YX', 'Y', 'Y'],['YX', 'XY', 'Y', 'XY', 'ZY'],['XY', 'Y', 'ZY', 'ZY', 'ZY']]
A_DC_term_CD_mixer_X = [['YX', 'ZY', 'YX', 'ZY', 'ZY'],['YX', 'ZY', 'Y', 'XY', 'ZY'],['YX', 'YZ', 'XY', 'Y', 'Y'],['XY', 'Y', 'ZY', 'Y', 'Y']]
A_DC_term_CD_prob_X = [['YX', 'ZY', 'XY', 'YX', 'YX'],['XY', 'ZY', 'XY', 'XY', 'XY'],['YX', 'Y', 'Y', 'Y', 'XY'],['ZY', 'Y', 'XY', 'Y', 'Y']]

# The variational parameter values taken from text files produced from running Schwinger_CD_algorithms.ipynb.
# For the circuits with an XY mixer.
QAOA_parameters_XY = [[[5.56746637,-0.57818399],[5.37515264, 0.21435273, -0.50880106, 0.29470067],[5.30979711, 0.1689656, 3.23346229, -0.63752236, 0.42775508, 5.64102399],[4.98227283, 0.04086656, 3.05570469, 4.47808945, -1.40638879, 0.39429014, 5.70970537, 3.23356511], [5.41929361, -0.04471674, 3.36653024, 4.19057482, 5.87911956, -1.41594864, 0.29435872, 5.52354058, 3.35873687, 1.15446697]], [[5.5832465, -0.5628222], [6.53933134, 5.33925742, -0.59840372, -0.01417669], [6.4707844, 5.3583176, 3.02417915, -0.58781017, -0.23730511, 5.33352081], [6.3011621, 5.38934639, 3.02036844, 3.11985037, -0.60691442, -0.34817669, 5.63004215, 4.98452006], [6.29458462, 5.37285349, 2.915258, 3.37373702, 2.98757524, -0.77345104, -0.29513524, 5.83870848, 5.10793712, 4.97593034]], [[0.68735581, 0.54978412], [0.84505846, 0.60468412, 0.29932494, 0.68276184], [0.89918436, 0.58974049, 0.44371615, 0.29032011, 0.47396192, 0.71101718], [0.69291946, 0.56816769, 0.2315899, 0.25360512, 0.2996151, 0.61824683, 0.91354667, 0.99259483], [0.82361775, 0.43181684, 0.17703929, 0.09244928, 0.32603651, 0.25282824, 0.6365772, 1.0000735, 1.10891812, 0.99506457]], [[5.58648674, -0.55799087], [6.63668629, 5.29823442, -0.55081751, -0.06602072], [6.90179769, 5.84830754, 0.88977109, 0.13404576, 0.47294231, 0.14649783], [7.18930173, 6.25808394, 0.4579868, 6.60164798, 0.44354044, 0.64059457, 0.05502922, 0.90453317], [7.19180859, 6.41228756, 0.5293176, 6.57734028, 0.18894186, 0.31664078, 0.47553573, 0.03811519, 0.65851982, 0.48773772]]]
DC_QAOA_parameters_XY = [[[-0.31567695, 5.66405316, -0.18357641],[-0.27292451, 6.07633206, 5.50616725, 3.09832902, -0.5965795, 5.22097204],[-0.15783116, 6.26616779, 2.99916084, 5.37822262, 2.83409216, 3.43503683, -0.83828854, 5.20177663, 5.33202055], [-0.01891663, 6.3405636, 3.06452511, 3.29172659, 5.25883273, 2.83766021, 3.31039515, 6.26727066, -1.26265201, 5.34933215, 5.19141025, 0.87678908],[-0.01982904, 6.25851519, 3.09669697, 3.09590582, 5.92075473, 5.01278583, 3.01869571, 3.21251044, 6.22237628, 4.29806792, -1.7208491,  5.34932892, 5.46357325, 0.84063532, 3.86686961]], [[0.27309664, 5.69376722, -0.24201019], [0.09675096, -0.26494852, 5.7644455, 6.35285592, -0.34467046, 0.40400265], [0.15003846, -0.09738891, 2.98851479, 5.41975971, 6.25595, 3.09643257, -0.56161379, 0.09856427, 5.20525609], [0.06452794, -0.05590436, 3.14688019, -0.11676545, 5.37132532, 6.19378984, 3.03273379, 3.15922789, -0.85585611, 0.05653695, 5.46521349, 5.03443515], [0.06067219, -0.05540759, 3.14993566, -0.0928096, 0.01252274, 5.36334241, 6.23289057, 3.01519575, 3.17935709, 6.27971853, -0.89115169, -0.01273437, 5.55520136, 4.95297623, 0.31215194]], [[6.4099642, 0.68124123, 0.40843171], [6.33133199, 2.95366361, 0.8135464, 5.98787051, 0.2939453, -0.3486224], [6.33998016, 3.00567949, 6.11227105, 0.72541571, 6.14253306, 6.21575741, 0.35526229, -0.5594, 0.34911344], [6.31241268, 3.03707192, 6.11557743, -0.09492101, 0.76752938, 6.11408525, 6.24859866, 6.19932834, 0.3488206, -0.62769791, 0.29306947, 0.31476292], [6.31154591, 3.0247575, 6.11022402, -0.09415251, 3.12057726, 0.7728715, 6.10178267, 6.23537722, 6.14389258, 6.3375187, 0.35509161, -0.5832378, 0.28654806, 0.31160139, 0.44401082]], [[6.35543437, 5.57382893, -0.48114962], [6.31281763, 6.00313638, 5.77151492, 6.30760205, -0.58174227, 0.57460004], [6.29721496, 6.09745683, 3.33757721, 5.86371512, 6.30557053, 6.24848418, -0.6493374, 0.61848809, -0.01149285], [6.30997701, 6.13876194, 3.26732499, 3.18983511, 6.05575609, 6.41761292, 6.39115513, 0.51006472, -0.27522691, 0.3492683, -0.05372919, 0.75188604], [6.29656982, 6.14379, 3.303669, 3.14791296, 6.23154493, 5.97685043, 6.36625118, 6.24329444, 0.67514939, 5.84526923, -0.42829815, 0.38641965, 0.0108004, 0.46062679, -0.20472428]]]
CD_inspired_parameters_XY = [[[-0.39305591],[-0.43690083, 3.49062466],[-0.05841762, 3.95489531, 2.69979322],[-0.06849813, 3.94337572, 2.66504193, 6.33784181],[-0.06970972, 3.9402824, 2.65798124, 6.29760324, 0.04994124]], [[-0.37860846], [-0.19254446, 3.37109214], [-0.14196819, 3.31833065, -0.12753316], [-0.12382064, 3.30266781, -0.08636592, 6.2047863], [-0.07326251, 3.27044987, -0.08312474, 6.20789429, 3.22768719]], [[-0.33484374], [-0.25525975, 6.05474669], [-0.10475015, 6.18367003, 3.35331555], [-0.09126231, 6.23605259, 3.32882533, 3.02704632], [0.20257478, 6.45830558, 3.44563588, 2.68059166, 3.24335639]], [[6.01466146], [6.09948021, 0.21085515], [6.15302786, 0.18837916, -0.11450702], [6.1567836, 0.18432481, -0.06999815, 3.08850076], [6.15450091, 0.1866346, -0.08004374, 3.06892057, 0.0341018]]]
CD_mixer_parameters_XY = [[[3.52464868e+00, 1.29277956e-03],[3.30782958e+00, -2.55772555e-01, 1.29899840e-03, 6.27580785e+00],[3.27603970e+00, -1.66207428e-01, 3.26970400e+00, 1.30664248e-03, 6.42885259e+00, 6.20206777e+00],[3.23501744e+00, 9.38377566e-01, 3.30400012e+00, 4.97750744e+00, 1.31040551e-03, 6.36237369e+00, 6.28307812e+00, 3.14064822e+00],[3.14514450e+00, 1.31730387e+00, 3.19524372e+00, 4.67629714e+00, 2.90200083e+00, 1.30778635e-03, 6.35206877e+00, 6.27349500e+00, 3.13254584e+00, 3.13916848e+00]], [[2.76221839, 0.27453571], [2.89931215, 6.05379461, 0.26093094, -0.00650956], [3.01371419, 6.19887388, 3.3556623, 0.24117113, 0.11516692, 6.20743104], [3.03378207, 6.20135808, 3.31263403, 6.355823, 0.22955368, 0.06725108, 6.24149532, 6.27991437], [3.03854894, 6.20300239, 3.28105307, 6.32809347, 0.06877732, 0.21633634, 0.05540335, 6.22312355, 6.39815929, 6.18676373]], [[-0.33659644, 5.9281659], [-0.18800362, 3.36366393, 5.99332383, 6.27930044], [-0.13393922, 3.3277998, 6.16084553, 6.04358967, 6.28407382, 6.27856161], [-0.10114585, 3.29639056, 6.15791109, 0.06587655, 6.06266778, 6.27044218, 6.27269056, 0.00724641], [-0.10763566, 3.29529776, 6.16519315, 0.02569941, 6.32976604, 6.07115697, 6.32625304, 6.23787308, -0.2718811, 0.28037805]], [[0.26023462, 5.86175702], [0.17210106, 2.91911629, 6.02363089, 6.27244917], [0.12409353, 2.94155165, 3.24723291, 6.09909862, 6.28316791, 6.27753657], [0.12464054, 2.9417114, 3.15850702, 0.09067735, 6.09391204, 6.25733309, 6.00979093, 0.27788659], [0.12546153, 2.93815216, 3.1806403, -0.05287684, 3.2634472, 6.09784342, 6.262245, 5.94637465, 0.18911221, 0.15649932]]]
CD_prob_parameters_XY = [[[5.89013131e+00, -2.47854995e-03],[5.86489976, 6.1460669, -0.00631368, 0.03293813],[6.02526166e+00, 6.22488194e+00, 2.92759486e+00, -1.30175616e-01, 2.45136764e-03, 2.34585078e-01], [6.0767007, 6.32135312, 2.80434798, 3.26642542, 0.04623959, -0.12014869, 0.12311286, 0.62158023], [6.04113078, 6.2869543, 2.85259188, 3.25780641, 3.15398563, 0.01345633, -0.0970041, 0.12677312, 0.59936832, 0.07699407]], [[-0.37860205, -0.00497707], [-0.23132181, 2.89604967, -0.10490461, 0.13846334], [-0.16585472, 3.07290192, 0.24581518, -0.0105789, -0.22613359, 0.276998], [-0.14003282, 3.0702556, 0.15712154, 3.33476693, -0.03431692, -0.34869059, 0.19028087, 0.32661146], [-0.09403445, 3.05565339, 0.09131709, 3.31394703, -0.16507567, -0.0563479, -0.41267114, 0.12601138, 0.34480674, 0.09733625]], [[2.80674202, -0.00493511], [2.95336349e+00, 3.36378180e+00, -4.45758347e-03, -2.03519809e-03], [3.03168899, 3.31012046, 6.1067648, -0.1569154, 0.11829212, 0.09422539], [3.039121, 3.25084891, 6.10713416, 0.15262542, -0.29746297, 0.04346826, 0.28030055, 0.04351896], [3.03952585, 3.24355461, 6.10006303, 0.16087539, 3.15677795, -0.32458581, 0.02605069, 0.31606923, 0.01981018, 0.15985911]], [[3.4000978, -0.00499999], [3.31414226e+00, 6.06040297e+00, -3.94662560e-03, -4.14449032e-03], [3.26701829e+00, 6.08239168e+00, 1.03765696e-01, -7.47416106e-03, 1.74930410e-04, 2.28289682e-03], [3.24336765, 6.1106116, 0.07881859, 6.39445668, 0.08004847, -0.05885813, -0.17748228, 0.15967789], [3.23792537, 6.16840323, 0.08683861, 6.34901636, 2.9908932, 0.22247054, -0.02256229, -0.26382673, 0.06246203, -0.07455539]]]

# For the circuits with an X mixer.
QAOA_parameters_X = [[[2.4123595, 5.82306032], [2.18759288, 0.809092, 6.19389196, 0.65737131], [2.27106032, 1.06411225, 1.68806472, 6.96766946, 0.76263751, 2.79991919], [2.24402237, 1.12961072, 1.86915944, 2.6942203, 7.06367329, 0.84754823, 2.67452731, 0.76485262], [2.51842384, 1.1156985, 1.68849513, 2.57035466, 1.6908483, 7.40193052, 0.77228277, 1.96707467, 0.99386528, 2.38966481]], [[5.68607648, 6.53116818], [5.95618033, 5.94200656, 6.41987118, 6.53343543], [6.08009681, 5.56414956, 1.78622087, 6.40640302, 6.81000697, 1.81559419], [6.08149805, 5.52071492, 2.24873523, 5.78512182, 6.43578364, 6.75537466, 1.95260357, 0.21917121], [6.13492671, 5.54525444, 2.3578333, 5.55867672, 3.39791001, 6.42313271, 6.81836476, 1.84108055, 0.39243337, 0.41486489]], [[5.55780416, 6.19970555], [6.07313926, 2.91697697, 7.01909448, 1.411925], [6.07230135, 2.92063702, 3.128941, 7.01999638, 1.40030857, 0.22293885], [6.29408781, 3.15971934, 3.3315661, 5.69277198, 6.93464591, 1.37756338, 0.45175261, 0.50059618], [6.2803998, 3.08886514, 3.45308266, 6.12067665, 2.64029806, 6.98710263, 1.44374384, 0.38132621, 0.28807957, 0.30635855]], [[0.75482283, 2.36444462], [0.51065758, 3.46399698, 2.43938185, 0.08900515], [0.3720697, 3.43093582, 2.56035499, 2.49451469, -0.05225227, 0.49945482], [0.30358471, 3.51742159, 2.56147294, 3.24936003, 2.50166034, -0.04613069, 0.47082267, 5.21023921], [-0.0137943, 3.15970204, 3.08261779, 2.34648301, 6.19418653, 2.40474956, 0.38568323, 0.29195731, 5.37734393, 2.63882369]]]
DC_QAOA_parameters_X = [[[1.8342512, 5.67364782, -0.13194502], [1.46966742, 5.37193963, 5.44290506, 1.49591295, -0.26817485, -0.0632905], [1.56565745, 5.36254539, 2.74606354, 5.49882346, 1.42116753, 0.82552793, 0.02909839, -0.03075204, 0.0389541], [1.57661284, 5.3718918, 2.64595298, 1.54290368, 5.48280837, 1.31608451, 0.76125092, 1.5913536, 0.01895847, -0.09699974, -0.09596787, 0.55469381], [1.56526316, 5.51212323, 3.59921605, 1.56366675, 4.6761989, 5.47089565, 1.33184797, 0.703492, 1.57657855, 6.28481116, -0.03973915, 0.07863742, 0.07472788, 0.20820876, 5.2034415]], [[1.4756777, 4.99544053, 0.41301922], [1.54495384, 0.72268748, 4.88365421, 3.39709355, 0.16391157, -0.14518545], [1.53833459, 0.82227466, 6.27314957, 4.76190256, 3.34851986, 6.25898119, 0.40040139, -0.45091459, 1.01485446], [1.56618345e+00, 8.36297479e-01, 5.46730861e+00, 2.87642693e-01, 4.71961177e+00, 3.42919739e+00, 6.39532806e+00, 5.40301371e+00, 3.35733840e-01, -3.68341302e-01, 4.32852104e-01, -5.88763136e-03], [1.56830466, 0.77660934, 5.4131133, 0.25061525, 2.97192255, 4.76151108, 3.52162127, 6.36277696, 5.45445232, 3.16795552, 0.22489161, -0.17758379, 0.13948095, -0.03597919, -0.34963435]], [[4.85985569, 3.45033752, 0.05954711], [4.85449069, 3.28749028, 3.41906631, -0.17205662, 0.13899907, 0.64159514], [4.82407004, 3.276746, 6.05818567, 3.44009086, 0.0325948, 3.10552807, 0.14562415, -0.0597997, 0.57880926], [4.74984446, 3.17855335, 5.71811478, 3.17016351, 3.40160765, 0.02642069, 3.42631119, 3.14170217, 0.12177477, -0.41057784, 0.37713764, 0.6753034], [4.64865108, 3.1596384, 5.12800313, 3.12884619, 1.55656022, 3.35609603, -0.0403279, 2.86301834, 2.98617199, 1.60996807, 0.13506931, -0.3159344, 0.12964774, 0.8118075, 0.03466943]], [[4.71151901, 6.51336903, 0.19627552], [4.71733727e+00, 3.91757704e+00, 6.56535696e+00, 1.60529478e+00, 6.61686622e-02, 4.34877159e-03], [4.70309049, 3.94061276, 1.57035017, 6.57795378, 1.52019027, 1.53250827, -0.0303235, -0.02219237, 0.70102437], [4.77294464, 3.84031974, 1.54379056, 4.70483852, 6.68629998, 1.73473479, 1.54856096, 3.13680282, -0.04468613, -0.0248576, -0.31445578, 2.6407078], [4.74319408, 3.84338319, 1.50113131, 4.7050228, 1.56329287, 6.63736799, 1.76414039, 1.54692947, 3.1485815, 3.14138278, 0.05463156, 0.01287748, -0.21035698, 1.77745543, 1.09699321]]]
CD_inspired_parameters_X = [[[1.80206971], [1.70849273, 2.26069581], [1.59582287, 2.20443562, 6.59725186], [1.58006837, 2.194478, 6.48587036, 0.17637475], [1.54224295, 1.58667375, 6.40631506, 0.31963789, 0.94301335]], [[1.73346124], [1.646242, 2.1677782], [1.57719366, 2.23516326, 3.47769114], [1.57725698, 2.23337604, 3.47644649, 3.15832936], [1.57725568, 2.23338968, 3.47645941, 2.72070397, 0.43756458]], [[4.96246568], [4.8532905, 0.80342992], [4.79691315, 0.92009679, 5.37370215], [4.71999988, 0.80306671, 5.39711115, 3.28838214], [4.72131265, 0.80692021, 5.47383412, 3.25818396, -0.12957537]], [[4.72502783], [4.70553449, 3.9191878], [4.82074428, 3.47962435, 4.12014987], [4.82134256, 3.48317235, 4.12042677, 3.08631412], [4.82145556, 3.48485462, 4.1205518, 3.10622346, -0.03513028]]]
CD_mixer_parameters_X = [[[1.80206971, 1.51276231],[1.42173695, 5.41278545, 1.51276231, 1.56588295], [1.59414817, 5.34720008, 0.31980465, 1.51276231, 1.56597888, 4.70586002], [1.54529424, 4.71648223, 0.44693478, 0.98133365, 1.51276231, 1.55083131, 4.72059972, 3.13921535], [1.54541954, 4.72047836, 0.4617176, 1.0097156, 2.99408695, 1.51276231, 1.56761681, 4.71000052, 3.13597785, 3.13568364]], [[1.73346124, 3.98952962], [1.48448289, -0.96393887, 3.98952962, 4.70738898], [1.48752039, -0.98184953, 1.49442831, 3.98952962, 4.70732752, 4.70732041], [1.54124088, -0.87430568, 1.51455682, 2.85366899, 3.98952962, 4.70781592, 4.68764672, 6.25542262], [1.54159783e+00, -8.75056378e-01, 1.47346620e+00, 2.85272849e+00, 3.25322009e-02, 3.98952962e+00, 4.70706628e+00, 4.67755583e+00, 6.25109475e+00, -4.65160828e-03]], [[4.8697596, 3.47678898], [4.75995708, 3.9754826, 3.47678898, 0.78043232], [4.71151942e+00, 4.06230890e+00, 5.81228422e+00, 3.47678898e+00, 7.79843077e-01, 2.69580414e-03], [4.73222226e+00, 4.04014615e+00, 5.81693739e+00, 6.28749000e+00, 3.47678898e+00, 7.78031977e-01, 2.74734849e-03, 3.30179500e+00], [4.7317157, 4.04292352, 5.81925157, 6.50342213, 2.92551048, 3.47678898, 0.77780511, 0.00657326, 3.31191419, 6.28167401]], [[4.72502783, 3.34923996], [4.70816662, 0.78212145, 3.34923996, 4.74451953], [4.82469088, 1.22295113, 2.15417933, 3.34923996, 4.71124267, 6.27828398], [4.82429952, 1.22369343, 2.1528549, 6.31204318, 3.34923996, 4.71171222, 6.27822736, 6.27817466], [4.82426687, 1.22375519, 2.15275632, 7.05721346, 2.21762394, 3.34923996, 4.71086252, 6.27832225, 6.15148474, 5.34169239]]]
CD_prob_parameters_X = [[[1.80207021e+00, -3.54933130e-03], [1.70838990e+00, 2.25998950e+00, -4.92487612e-03, -4.89142134e-03], [1.59278834, 2.25413875, 2.7995662, 0.0358313, -0.0283619,  0.06056121], [1.58412732, 2.366958, 1.9120874, 0.0885791, -0.09798128, 0.00774373, 0.10711318, 5.65808942], [1.56309624, 2.3507509, 1.1170291, -0.02260503, 4.67314253, -0.0112494, -0.01947405, -0.06779229, 5.56933434, 5.53139767]], [[1.57355391e+00, -2.73531919e-03], [1.58426580e+00, 1.04113780e+00, -1.63509896e-03, -4.31257958e-03], [1.54372615, 0.86595406, 5.90048416, 0.01233457, -0.01860945, 0.03164848], [1.55362202e+00, 8.89579885e-01, 6.03100640e+00, 2.94986867e+00, -8.72388441e-03,  5.35548785e-03, -7.63477461e-02, 1.27972478e-01], [1.58345005e+00, 8.49139867e-01, 5.50258833e+00, 3.10159265e+00, 6.25233754e+00, -2.35782945e-03, -2.76154304e-02, -1.38118311e-02, -5.45169490e-01, 1.07091625e+00]], [[4.80580106, -0.005], [4.80246261, 5.81397483, -0.01099763, -0.00935687], [4.76959516, 5.92071761, 2.5149002, 0.08905098, -0.31108393, 0.6971442], [4.80839894, 6.15073321, 2.77625183, 3.13511009, -0.03241166, -0.50198225, 0.48073327, 0.5812473], [4.74406846, 6.15108343, 2.76113989, 3.17468976, -0.0138078, -0.03758272, -0.49358102, 0.49785628, 0.49761424, 0.33192849]], [[2.38691277, 6.12279417], [2.38350695, 2.97686299, 6.13592076, -0.05805014], [2.39773619, 3.14382178, 4.77787065, 6.05418042, 0.06714699, 3.94598361], [2.3921817, 3.12582754, 4.77481942, 3.075969, 6.02275102, 0.06499035, 3.92611711, 1.08163297], [2.39890183, 3.12052593, 4.77316249, 3.03586596, 3.13373496, 6.06711621, -0.04034025, 4.05994399, 0.86271235, 0.66638224]]]

# String of the type of plot to be used in the text file.
plot_type = 'CD algorithms resource estimation'
# String of the name of the code to be recorded in text file.
code_name = 'Schwinger_CD_algorithms_resource_estimation.ipynb'

In [ ]:

# String of the model parameters for the text file.
plot_parameters = rf"$a={a_string}, g={g}, J={J:.2}, w={w:.2f}, m={m:.2f}, \theta={theta}$"
# Clean variables have \'s and $'s (which are needed for latex writing in plots) removed for the text file.
plot_parameters_clean = plot_parameters.replace("\\", "").replace("$", "")

# Importing and setting FakeAlgiers as the backend
from qiskit_ibm_runtime.fake_provider import FakeAlgiers
backend = FakeAlgiers()

# Stores the counts of the different operations, total number of operations, and the circuit depth for each of the algorithms with a certain mixer.
count_ops_full_QAOA_XY = []
size_full_QAOA_XY = []
depth_full_QAOA_XY = []

count_ops_full_QAOA_X = []
size_full_QAOA_X = []
depth_full_QAOA_X = []

count_ops_full_DC_QAOA_XY = []
size_full_DC_QAOA_XY = []
depth_full_DC_QAOA_XY = []

count_ops_full_DC_QAOA_X = []
size_full_DC_QAOA_X = []
depth_full_DC_QAOA_X = []

count_ops_full_CD_inspired_XY = []
size_full_CD_inspired_XY = []
depth_full_CD_inspired_XY = []

count_ops_full_CD_inspired_X = []
size_full_CD_inspired_X = []
depth_full_CD_inspired_X = []

count_ops_full_CD_mixer_XY = []
size_full_CD_mixer_XY = []
depth_full_CD_mixer_XY = []

count_ops_full_CD_mixer_X = []
size_full_CD_mixer_X = []
depth_full_CD_mixer_X = []

count_ops_full_CD_prob_XY = []
size_full_CD_prob_XY = []
depth_full_CD_prob_XY = []

count_ops_full_CD_prob_X = []
size_full_CD_prob_X = []
depth_full_CD_prob_X = []

# Indexes the A_DC operators and parameter values for different qubits.
count = 0

# Loop over the different numbers of qubits.
for num_qubits in num_qubits_list:

    # Constructing the Schwinger model Hamiltonian and the mixers.
    H_prob_Schwinger = Schwinger_Hamiltonian(num_qubits, J, w, m, theta)
    H_mix_XY, initial_state_XY = XY_mixer(num_qubits)
    H_mix_X, initial_state_X = X_mixer(num_qubits)

    # Stores the counts of the different operations, total number of operations, and the circuit depth for each of the algorithms 
    # with a certain mixer for a layer.
    count_ops_layers_QAOA_XY = []
    size_layers_QAOA_XY = []
    depth_layers_QAOA_XY = []

    count_ops_layers_QAOA_X = []
    size_layers_QAOA_X = []
    depth_layers_QAOA_X = []

    count_ops_layers_DC_QAOA_XY = []
    size_layers_DC_QAOA_XY = []
    depth_layers_DC_QAOA_XY = []

    count_ops_layers_DC_QAOA_X = []
    size_layers_DC_QAOA_X = []
    depth_layers_DC_QAOA_X = []

    count_ops_layers_CD_inspired_XY = []
    size_layers_CD_inspired_XY = []
    depth_layers_CD_inspired_XY = []

    count_ops_layers_CD_inspired_X = []
    size_layers_CD_inspired_X = []
    depth_layers_CD_inspired_X = []

    count_ops_layers_CD_mixer_XY = []
    size_layers_CD_mixer_XY = []
    depth_layers_CD_mixer_XY = []

    count_ops_layers_CD_mixer_X = []
    size_layers_CD_mixer_X = []
    depth_layers_CD_mixer_X = []

    count_ops_layers_CD_prob_XY = []
    size_layers_CD_prob_XY = []
    depth_layers_CD_prob_XY = []

    count_ops_layers_CD_prob_X = []
    size_layers_CD_prob_X = []
    depth_layers_CD_prob_X = []

    # Stores the A_DC operators for each algorithm with a certain mixer.
    A_DC_DC_QAOA_XY = []
    A_DC_CD_inspired_XY = []
    A_DC_CD_mixer_XY = []
    A_DC_CD_prob_XY = []
    
    A_DC_DC_QAOA_X = []
    A_DC_CD_inspired_X = []
    A_DC_CD_mixer_X = []
    A_DC_CD_prob_X = []

    # Loops over the number of layers.
    for i in layers:

        # Runs QAOA with an XY mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = QAOA(H_prob_Schwinger, H_mix_XY, initial_state_XY, i, num_qubits, QAOA_parameters_XY[count][i-1])
        count_ops_layers_QAOA_XY.append(count_ops)
        size_layers_QAOA_XY.append(size)
        depth_layers_QAOA_XY.append(depth)

        # Runs QAOA with an X mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = QAOA(H_prob_Schwinger, H_mix_X, initial_state_X, i, num_qubits, QAOA_parameters_X[count][i-1])
        count_ops_layers_QAOA_X.append(count_ops)
        size_layers_QAOA_X.append(size)
        depth_layers_QAOA_X.append(depth)

        # Appends the A_DC operator for DC-QAOA with an XY mixer for this layer.
        A_DC_DC_QAOA_XY.append(CD_Hamiltonian_XY(num_qubits, A_DC_term_DC_QAOA_XY[count][i-1]))
        # Runs DC-QAOA with an XY mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = DC_QAOA(H_prob_Schwinger, H_mix_XY, A_DC_DC_QAOA_XY, initial_state_XY, i, num_qubits, DC_QAOA_parameters_XY[count][i-1])
        count_ops_layers_DC_QAOA_XY.append(count_ops)
        size_layers_DC_QAOA_XY.append(size)
        depth_layers_DC_QAOA_XY.append(depth)

        # Appends the A_DC operator for DC-QAOA with an X mixer for this layer.
        A_DC_DC_QAOA_X.append(CD_Hamiltonian_X(num_qubits, A_DC_term_DC_QAOA_X[count][i-1]))
        # Runs DC-QAOA with an X mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = DC_QAOA(H_prob_Schwinger, H_mix_X, A_DC_DC_QAOA_X, initial_state_X, i, num_qubits, DC_QAOA_parameters_X[count][i-1])
        count_ops_layers_DC_QAOA_X.append(count_ops)
        size_layers_DC_QAOA_X.append(size)
        depth_layers_DC_QAOA_X.append(depth)

        # Appends the A_DC operator for CD-inspired with an XY mixer for this layer.
        A_DC_CD_inspired_XY.append(CD_Hamiltonian_XY(num_qubits, A_DC_term_CD_inspired_XY[count][i-1]))
        # Runs CD-inspired with an XY mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_inspired(H_prob_Schwinger, A_DC_CD_inspired_XY, initial_state_XY, i, num_qubits, CD_inspired_parameters_XY[count][i-1])
        count_ops_layers_CD_inspired_XY.append(count_ops)
        size_layers_CD_inspired_XY.append(size)
        depth_layers_CD_inspired_XY.append(depth)

        # Appends the A_DC operator for CD-inspired with an X mixer for this layer.
        A_DC_CD_inspired_X.append(CD_Hamiltonian_X(num_qubits, A_DC_term_CD_inspired_X[count][i-1]))
        # Runs CD-inspired with an X mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_inspired(H_prob_Schwinger, A_DC_CD_inspired_X, initial_state_X, i, num_qubits, CD_inspired_parameters_X[count][i-1])
        count_ops_layers_CD_inspired_X.append(count_ops)
        size_layers_CD_inspired_X.append(size)
        depth_layers_CD_inspired_X.append(depth)

        # Appends the A_DC operator for CD-mixer with an XY mixer for this layer.
        A_DC_CD_mixer_XY.append(CD_Hamiltonian_XY(num_qubits, A_DC_term_CD_mixer_XY[count][i-1]))
        # Runs CD-mixer with an XY mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_mixer(H_prob_Schwinger, H_mix_XY, A_DC_CD_mixer_XY, initial_state_XY, i, num_qubits, CD_mixer_parameters_XY[count][i-1])
        count_ops_layers_CD_mixer_XY.append(count_ops)
        size_layers_CD_mixer_XY.append(size)
        depth_layers_CD_mixer_XY.append(depth)

        # Appends the A_DC operator for CD-mixer with an X mixer for this layer.
        A_DC_CD_mixer_X.append(CD_Hamiltonian_X(num_qubits, A_DC_term_CD_mixer_X[count][i-1]))
        # Runs CD-mixer with an X mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_mixer(H_prob_Schwinger, H_mix_X, A_DC_CD_mixer_X, initial_state_X, i, num_qubits, CD_mixer_parameters_X[count][i-1])
        count_ops_layers_CD_mixer_X.append(count_ops)
        size_layers_CD_mixer_X.append(size)
        depth_layers_CD_mixer_X.append(depth)

        # Appends the A_DC operator for CD-prob with an XY mixer for this layer.
        A_DC_CD_prob_XY.append(CD_Hamiltonian_XY(num_qubits, A_DC_term_CD_prob_XY[count][i-1]))
        # Runs CD-prob with an XY mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_prob(H_prob_Schwinger, A_DC_CD_prob_XY, initial_state_XY, i, num_qubits, CD_prob_parameters_XY[count][i-1])
        count_ops_layers_CD_prob_XY.append(count_ops)
        size_layers_CD_prob_XY.append(size)
        depth_layers_CD_prob_XY.append(depth)

        # Appends the A_DC operator for CD-prob with an X mixer for this layer.
        A_DC_CD_prob_X.append(CD_Hamiltonian_X(num_qubits, A_DC_term_CD_prob_X[count][i-1]))
        # Runs CD-prob with an X mixer and stores the counts of the different operations, total number of operations, and the circuit depth.
        count_ops, size, depth = CD_prob(H_prob_Schwinger, A_DC_CD_prob_X, initial_state_X, i, num_qubits, CD_prob_parameters_X[count][i-1])
        count_ops_layers_CD_prob_X.append(count_ops)
        size_layers_CD_prob_X.append(size)
        depth_layers_CD_prob_X.append(depth)

    # Appends the counts of the different operations, total number of operations, and the circuit depth.
    count_ops_full_QAOA_XY.append(count_ops_layers_QAOA_XY)
    size_full_QAOA_XY.append(size_layers_QAOA_XY)
    depth_full_QAOA_XY.append(depth_layers_QAOA_XY)

    count_ops_full_QAOA_X.append(count_ops_layers_QAOA_X)
    size_full_QAOA_X.append(size_layers_QAOA_X)
    depth_full_QAOA_X.append(depth_layers_QAOA_X)

    count_ops_full_DC_QAOA_XY.append(count_ops_layers_DC_QAOA_XY)
    size_full_DC_QAOA_XY.append(size_layers_DC_QAOA_XY)
    depth_full_DC_QAOA_XY.append(depth_layers_DC_QAOA_XY)

    count_ops_full_DC_QAOA_X.append(count_ops_layers_DC_QAOA_X)
    size_full_DC_QAOA_X.append(size_layers_DC_QAOA_X)
    depth_full_DC_QAOA_X.append(depth_layers_DC_QAOA_X)

    count_ops_full_CD_inspired_XY.append(count_ops_layers_CD_inspired_XY)
    size_full_CD_inspired_XY.append(size_layers_CD_inspired_XY)
    depth_full_CD_inspired_XY.append(depth_layers_CD_inspired_XY)

    count_ops_full_CD_inspired_X.append(count_ops_layers_CD_inspired_X)
    size_full_CD_inspired_X.append(size_layers_CD_inspired_X)
    depth_full_CD_inspired_X.append(depth_layers_CD_inspired_X)
    
    count_ops_full_CD_mixer_XY.append(count_ops_layers_CD_mixer_XY)
    size_full_CD_mixer_XY.append(size_layers_CD_mixer_XY)
    depth_full_CD_mixer_XY.append(depth_layers_CD_mixer_XY)

    count_ops_full_CD_mixer_X.append(count_ops_layers_CD_mixer_X)
    size_full_CD_mixer_X.append(size_layers_CD_mixer_X)
    depth_full_CD_mixer_X.append(depth_layers_CD_mixer_X)

    count_ops_full_CD_prob_XY.append(count_ops_layers_CD_prob_XY)
    size_full_CD_prob_XY.append(size_layers_CD_prob_XY)
    depth_full_CD_prob_XY.append(depth_layers_CD_prob_XY)

    count_ops_full_CD_prob_X.append(count_ops_layers_CD_prob_X)
    size_full_CD_prob_X.append(size_layers_CD_prob_X)
    depth_full_CD_prob_X.append(depth_layers_CD_prob_X)

    count += 1

# Add timestamp to plot file name to prevent plots from being saved over.
timestamp_EP_full = datetime.now().strftime("%Y%m%d_%H%M%S")
# Filename of the text file.
filename_subplot_EP_full = rf"C:\Users\ebl1v24\OneDrive - University of Southampton\PhD\DC-QAOA project\Plots\Schwinger model {plot_type}_{timestamp_EP_full}"

# Creates text file to store the data..
with open(filename_subplot_EP_full + ".txt", "w") as f:
    
    f.write(rf"Schwinger model {plot_type}, {plot_parameters_clean}, FakeAlgiers backend.")
    f.write("\n\n")
    
    f.write("=" * 50 + "\n")
    f.write(f"XY mixer\n")
    f.write("=" * 50 + "\n\n")

    for i in range(len(num_qubits_list)):

        f.write("-" * 50 + "\n")
        f.write(f"{num_qubits_list[i]} qubits \n")
        f.write("-" * 50 + "\n\n")

        for j in range(layers[-1]):

            f.write("-" * 50 + "\n")
            f.write(f"Layer = {j+1}\n")
            f.write("-" * 50 + "\n\n")

            f.write('QAOA:' + '\n\n')
            f.write(f'Operators: {count_ops_full_QAOA_XY[i][j]} \n')
            f.write(f'Operations: {size_full_QAOA_XY[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_QAOA_XY[i][j]} \n\n')

            f.write('DC-QAOA:' + '\n\n')
            f.write(f'Operators: {count_ops_full_DC_QAOA_XY[i][j]} \n')
            f.write(f'Operations: {size_full_DC_QAOA_XY[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_DC_QAOA_XY[i][j]} \n\n')

            f.write('CD-inspired:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_inspired_XY[i][j]} \n')
            f.write(f'Operations: {size_full_CD_inspired_XY[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_inspired_XY[i][j]} \n\n')

            f.write('CD-mixer:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_mixer_XY[i][j]} \n')
            f.write(f'Operations: {size_full_CD_mixer_XY[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_mixer_XY[i][j]} \n\n')

            f.write('CD-prob:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_prob_XY[i][j]} \n')
            f.write(f'Operations: {size_full_CD_prob_XY[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_prob_XY[i][j]} \n\n')

    f.write("=" * 50 + "\n")
    f.write(f"X mixer\n")
    f.write("=" * 50 + "\n\n")

    for i in range(len(num_qubits_list)):

        f.write("-" * 50 + "\n")
        f.write(f"{num_qubits_list[i]} qubits \n")
        f.write("-" * 50 + "\n\n")

        for j in range(layers[-1]):

            f.write("-" * 50 + "\n")
            f.write(f"Layer = {j+1}\n")
            f.write("-" * 50 + "\n\n")

            f.write('QAOA:' + '\n\n')
            f.write(f'Operators: {count_ops_full_QAOA_X[i][j]} \n')
            f.write(f'Operations: {size_full_QAOA_X[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_QAOA_X[i][j]} \n\n')

            f.write('DC-QAOA:' + '\n\n')
            f.write(f'Operators: {count_ops_full_DC_QAOA_X[i][j]} \n')
            f.write(f'Operations: {size_full_DC_QAOA_X[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_DC_QAOA_X[i][j]} \n\n')

            f.write('CD-inspired:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_inspired_X[i][j]} \n')
            f.write(f'Operations: {size_full_CD_inspired_X[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_inspired_X[i][j]} \n\n')

            f.write('CD-mixer:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_mixer_X[i][j]} \n')
            f.write(f'Operations: {size_full_CD_mixer_X[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_mixer_X[i][j]} \n\n')

            f.write('CD-prob:' + '\n\n')
            f.write(f'Operators: {count_ops_full_CD_prob_X[i][j]} \n')
            f.write(f'Operations: {size_full_CD_prob_X[i][j]} \n')
            f.write(f'Circuit depth: {depth_full_CD_prob_X[i][j]} \n\n')

            f.write('\n\n')
            

